In [1]:
!uv sync

Resolved 103 packages in 3ms
Checked 79 packages in 1ms


# 1. Loading and preparation

All experiment code uses reusable modules from `avito_retrieval`. The test set is loaded only for final inference; validation uses `calibration`.

In [2]:
import pandas as pd
from avito_retrieval import load_dataset

In [3]:
DATA_PATH = "candidate_public/candidate_data"

In [4]:
data = load_dataset(DATA_PATH)

/Users/waxyyy99/Code/avito_bootcamp_test/.venv/lib/python3.13/site-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


In [5]:
articles = data.articles
calibration = data.calibration
test = data.test

print(f"articles={len(articles)}, calibration={len(calibration)}, test={len(test)}")
articles[["article_id", "clean_title", "document_text"]].head()

articles=793, calibration=500, test=500


,article_id,clean_title,document_text
0,1730,имя или название компании,имя или название компании имя или название ком...
1,1746,"понять, что профиль заблокирован","понять, что профиль заблокирован понять, что п..."
2,1747,не допустить блокировки профиля,не допустить блокировки профиля не допустить б...
3,1774,оставить или удалить профиль,оставить или удалить профиль оставить или удал...
4,1775,удалить профиль,удалить профиль удалить профиль ⚡ удалить проф...


# 2. Lexical baseline

This ranker combines word TF-IDF query-to-query retrieval, character n-gram query-to-query retrieval, and direct word TF-IDF article retrieval. Every fold fits on its own calibration training split.

In [6]:
from avito_retrieval import evaluate_lexical_cv

lexical_result = evaluate_lexical_cv(calibration, articles)
print("fold MAP@10:", [round(score, 4) for score in lexical_result.fold_scores])
print(f"mean MAP@10: {lexical_result.mean_score:.4f}")

fold MAP@10: [0.5416, 0.5611, 0.5575, 0.5863, 0.593]
mean MAP@10: 0.5679


# 3. Local FRIDA embeddings

FRIDA is loaded from `models/frida` with `local_files_only=True`. It uses `paraphrase` for query-to-query retrieval and the asymmetric `search_query` / `search_document` prompts for query-to-article retrieval.

In [7]:
from avito_retrieval import FridaEmbedder

frida = FridaEmbedder(model_path="models/frida", batch_size=8)
sample_vectors = frida.encode(
    ["как вернуть деньги за товар", "когда вернут деньги после возврата"],
    prompt_name="paraphrase",
)
print(f"device={frida.device}, vectors={sample_vectors.shape}")
print(f"cosine similarity={sample_vectors[0] @ sample_vectors[1]:.4f}")

/Users/waxyyy99/Code/avito_bootcamp_test/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 219/219 [00:01<00:00, 124.77it/s]


device=mps, vectors=(2, 1536)
cosine similarity=0.8408
